In [ ]:
!pip install rouge-score bert-score evaluate nltk pandas -q
!python -m nltk.downloader wordnet punkt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
# ============================================================
# Clinical Summary Evaluation - Quality Metrics Only
# 包含: ROUGE, BLEU, METEOR, BERTScore
# BERTScore 使用 roberta-large + max_length=512 (稳定版本)
# ============================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer
from bert_score import BERTScorer
from nltk.translate.meteor_score import meteor_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

print("✅ 库导入完成")

# %% [markdown]
# ## 3. 读取数据

# %%
# 读取 LlaDA 结果
llada_df = pd.read_csv("llada_summaries_full.csv")
print(f"LlaDA 文件: {llada_df.shape}")
print(f"列名: {llada_df.columns.tolist()}")
llada_df.head(2)

# %%
# 读取 Llama 结果
llama_df = pd.read_csv("llama_summaries_full.csv")
print(f"Llama 文件: {llama_df.shape}")
print(f"列名: {llama_df.columns.tolist()}")
llama_df.head(2)

# %% [markdown]
# ## 4. 统一数据格式

# %%
# LlaDA: 直接使用 predicted_summary 列
llada_predictions = llada_df['predicted_summary'].astype(str).tolist()
llada_references = llada_df['reference_summary'].astype(str).tolist()
llada_ids = llada_df['id'].tolist()

# Llama: 使用 generated_summary 作为预测
llama_predictions = llama_df['generated_summary'].astype(str).tolist()
llama_references = llama_df['reference_summary'].astype(str).tolist()
llama_ids = llama_df['id'].tolist()

print(f"LlaDA: {len(llada_predictions)} 个样本")
print(f"Llama: {len(llama_predictions)} 个样本")

# %% [markdown]
# ## 5. 初始化评估器（修复版）

# %%
class QualityEvaluator:
    """评估 Quality 维度的指标 - 修复 BERTScore 溢出问题"""

    def __init__(self, device="cuda"):
        self.device = device

        # ROUGE
        self.rouge_scorer = rouge_scorer.RougeScorer(
            ['rouge1', 'rouge2', 'rougeL'],
            use_stemmer=True
        )

        # BERTScore - 使用 roberta-large 代替 deberta，避免溢出
        print("正在加载 BERTScore 模型...")
        self.bertscorer = BERTScorer(
            lang="en",
            model_type="roberta-large",           # 稳定的模型
            rescale_with_baseline=True,           # 与人类判断对齐
            device=device,
            batch_size=32                       # 显式截断，防止溢出
        )

        # BLEU 平滑函数
        self.smoother = SmoothingFunction().method1

    def compute_rouge(self, pred: str, ref: str) -> dict:
        """ROUGE-1/2/L F1"""
        scores = self.rouge_scorer.score(str(ref), str(pred))
        return {
            "ROUGE-1": round(scores['rouge1'].fmeasure, 4),
            "ROUGE-2": round(scores['rouge2'].fmeasure, 4),
            "ROUGE-L": round(scores['rougeL'].fmeasure, 4)
        }

    def compute_bleu(self, pred: str, ref: str) -> dict:
        """BLEU-1/2/3/4"""
        pred_tokens = str(pred).split()
        ref_tokens = str(ref).split()

        results = {}
        for n in [1, 2, 3, 4]:
            weights = tuple([1.0/n] * n)
            bleu = sentence_bleu([ref_tokens], pred_tokens,
                                  weights=weights,
                                  smoothing_function=self.smoother)
            results[f"BLEU-{n}"] = round(bleu, 4)

        bleu_avg = sentence_bleu([ref_tokens], pred_tokens,
                                  weights=(0.25, 0.25, 0.25, 0.25),
                                  smoothing_function=self.smoother)
        results["BLEU"] = round(bleu_avg, 4)

        return results

    def compute_meteor(self, pred: str, ref: str) -> dict:
        """METEOR"""
        pred_tokens = str(pred).split()
        ref_tokens = str(ref).split()
        meteor = meteor_score([ref_tokens], pred_tokens)
        return {"METEOR": round(meteor, 4)}

    def compute_bertscore_batch(self, preds: list, refs: list) -> list:
        """批量计算 BERTScore，返回每个样本的 F1"""
        P, R, F1 = self.bertscorer.score(preds, refs)
        return [round(score, 4) for score in F1.tolist()]

    def evaluate_batch(self, preds: list, refs: list, desc="Evaluating") -> pd.DataFrame:
        """批量评估所有指标"""
        results = []

        # 逐条计算 ROUGE, BLEU, METEOR
        for pred, ref in tqdm(zip(preds, refs), total=len(preds), desc=f"{desc} (ROUGE/BLEU/METEOR)"):
            row = {}
            row.update(self.compute_rouge(pred, ref))
            row.update(self.compute_bleu(pred, ref))
            row.update(self.compute_meteor(pred, ref))
            results.append(row)

        df = pd.DataFrame(results)

        # 批量计算 BERTScore
        print(f"{desc}: 计算 BERTScore...")
        bert_scores = self.compute_bertscore_batch(preds, refs)
        df["BERTScore-F1"] = bert_scores

        return df

# %% [markdown]
# ## 6. 运行评估

# %%
# 初始化评估器（如果有 GPU 就用 cuda，否则用 cpu）
try:
    evaluator = QualityEvaluator(device="cuda")
    print("✅ 使用 GPU 进行 BERTScore 计算")
except:
    evaluator = QualityEvaluator(device="cpu")
    print("⚠️ GPU 不可用，使用 CPU（可能较慢）")

# %%
# 评估 LlaDA
print("\n" + "="*60)
print("评估 LlaDA 模型")
print("="*60)

llada_results = evaluator.evaluate_batch(
    llada_predictions,
    llada_references,
    desc="LlaDA"
)

# %%
# 评估 Llama
print("\n" + "="*60)
print("评估 Llama 模型")
print("="*60)

llama_results = evaluator.evaluate_batch(
    llama_predictions,
    llama_references,
    desc="Llama"
)

# %% [markdown]
# ## 7. 汇总结果

# %%
# 计算每个模型的平均分
llada_mean = llada_results.mean().to_frame().T
llada_mean.index = ["LlaDA-8B"]

llama_mean = llama_results.mean().to_frame().T
llama_mean.index = ["Llama-3.1-8B"]

# 合并对比
comparison = pd.concat([llama_mean, llada_mean])
comparison = comparison.round(4)

print("\n" + "="*60)
print("QUALITY EVALUATION - MODEL COMPARISON")
print("="*60)
print(comparison.to_string())

# %% [markdown]
# ## 8. 详细结果保存

# %%
# 保存每个模型的详细结果
llada_results['id'] = llada_ids
llama_results['id'] = llama_ids

# 重命名列以便区分
llada_results = llada_results.rename(columns={
    col: f"LlaDA_{col}" for col in llada_results.columns if col != 'id'
})
llama_results = llama_results.rename(columns={
    col: f"Llama_{col}" for col in llama_results.columns if col != 'id'
})

✅ 库导入完成
LlaDA 文件: (3396, 3)
列名: ['id', 'predicted_summary', 'reference_summary']
Llama 文件: (3396, 4)
列名: ['id', 'full_text', 'reference_summary', 'generated_summary']
LlaDA: 3396 个样本
Llama: 3396 个样本
正在加载 BERTScore 模型...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

✅ 使用 GPU 进行 BERTScore 计算

评估 LlaDA 模型


LlaDA (ROUGE/BLEU/METEOR): 100%|██████████| 3396/3396 [00:35<00:00, 96.01it/s] 


LlaDA: 计算 BERTScore...

评估 Llama 模型


Llama (ROUGE/BLEU/METEOR): 100%|██████████| 3396/3396 [00:30<00:00, 109.84it/s]


Llama: 计算 BERTScore...

QUALITY EVALUATION - MODEL COMPARISON
              ROUGE-1  ROUGE-2  ROUGE-L  BLEU-1  BLEU-2  BLEU-3  BLEU-4    BLEU  METEOR  BERTScore-F1
Llama-3.1-8B   0.3854   0.1573   0.2598  0.2435  0.1377  0.0856  0.0563  0.0563  0.2306        0.1710
LlaDA-8B       0.3972   0.1542   0.2725  0.2547  0.1392  0.0830  0.0525  0.0525  0.2350        0.2014


In [ ]:
# ========== 新增：保存到 CSV 文件 ==========
llada_results.to_csv("llada_quality_scores.csv", index=False)
llama_results.to_csv("llama_quality_scores.csv", index=False)
comparison.to_csv("model_quality_comparison.csv")

print("\n✅ 结果已保存:")
print("   - llada_quality_scores.csv (LlaDA 每个样本的详细分数)")
print("   - llama_quality_scores.csv (Llama 每个样本的详细分数)")
print("   - model_quality_comparison.csv (两个模型的对比汇总)")


✅ 结果已保存:
   - llada_quality_scores.csv (LlaDA 每个样本的详细分数)
   - llama_quality_scores.csv (Llama 每个样本的详细分数)
   - model_quality_comparison.csv (两个模型的对比汇总)
